In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import math
import numpy as np
import pandas as pd
import xarray as xr
import networkx as nx
import plotly.express as px 
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from os.path import join as pjoin
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr, zscore, kendalltau
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/overrep_cell_corrs'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 0.2 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians
chunks = 4 ## number of time bins for cell-cell correlations

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

### Example mouse. Create a over-representation vs reward graph by binning the session into equal sized bins.

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'
port_type = 'Undershooting'
num_bins = 4
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)

In [ ]:
## Load and process data
output_dict = {'mouse': [], 'group': [], 'sex': [], 'bin': [], 'reward_zone': [], 'non_reward_zone': [], 'other_port': [], 'num_rewards': []}
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata.assign_coords(cumulative_rewards=('frame', np.cumsum(sdata['water'].values))) ## create cumulative rewards coordinate for later
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Get positions of the under or over-shooting ports and use those as the zero location
front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
if port_type == 'Overshooting':
    pt1_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
    pt2_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
elif port_type == 'Undershooting':
    pt1_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
    pt2_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)

num_frames = int(sdata.shape[1] / num_bins)
for rw_num in np.arange(1, num_bins + 1):
    sub_data = sdata[:, num_frames * (bin - 1):num_frames * bin]
    neural_data, position_data = ctn.subset_correct_dir_and_running(sub_data, correct_dir=correct_dir, only_running=only_running, 
                                                                    velocity_thresh=velocity_thresh)
    ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
    population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
    active_cells = np.sum(population_activity, axis=0) != 0
    population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
    tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
    tuning_curves = tuning_curves.T ## cells x spatial bin

    ## Find the peak of each place field
    pf_peaks = np.max(tuning_curves, axis=1)
    ## Find the spatial bins where each peak occurred
    field_dist = np.zeros(pf_peaks.shape[0])
    for idx, peak in enumerate(pf_peaks):
        field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]


    ## Make Reward 1 the first rewarding port the mouse got water from
    first_rew = sdata['lick_port'][sdata['water']].values[0]
    if first_rew == sdata.attrs['reward_one']:
        first_rw_pos = reward_one_pos 
        second_rw_pos = reward_two_pos
    else:
        first_rw_pos = reward_two_pos 
        second_rw_pos = reward_one_pos

    ## Find distance from reward locations
    rw_one_dist = field_dist - first_rw_pos
    rw_two_dist = field_dist - second_rw_pos
    bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
    sub_uids = neural_data['unit_id'][active_cells]
    rw1_uids = sub_uids['unit_id'][rw1_bool]
    rw2_uids = sub_uids['unit_id'][rw2_bool]

    ## Find distance from a an area close by
    ## Could play around by changing the index in all_midpoints, I chose 3 bins away because
    ## the build up of place fields can be wide
    non_rw_bin_start, non_rw_bin_end = all_midpoints[-1] - (reward_bin_size / 2), all_midpoints[-1] + (reward_bin_size / 2)
    non_rw1_bool = (rw_one_dist >= non_rw_bin_start) & (rw_one_dist < non_rw_bin_end)
    non_rw2_bool = (rw_two_dist >= non_rw_bin_start) & (rw_two_dist < non_rw_bin_end)
    non_rw1_uids = sub_uids['unit_id'][non_rw1_bool]
    non_rw2_uids = sub_uids['unit_id'][non_rw2_bool]

    ## Find distance from the undershooting or overshooting port
    pt_one_dist = field_dist - pt1_rw_pos
    pt_two_dist = field_dist - pt2_rw_pos
    bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
    pt1_bool = (pt_one_dist >= bin_start) & (pt_one_dist < bin_end)
    pt2_bool = (pt_two_dist >= bin_start) & (pt_two_dist < bin_end)
    pt1_uids = sub_uids['unit_id'][pt1_bool]
    pt2_uids = sub_uids['unit_id'][pt2_bool]

    output_dict['mouse'].append(mouse)
    output_dict['group'].append(sdata.attrs['group'])
    output_dict['sex'].append(sdata.attrs['sex'])
    output_dict['bin'].append(bin * num_frames)
    output_dict['reward_zone'].append((rw1_uids.shape[0] + rw2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['non_reward_zone'].append((non_rw1_uids.shape[0] + non_rw2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['other_port'].append((pt1_uids.shape[0] + pt2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['num_rewards'].append(np.max(sub_data['cumulative_rewards'].values))

rep_df = pd.DataFrame(output_dict)

In [ ]:
## Plot proportion of place fields across rewards
fig = pf.custom_graph_template(x_title='Number of Rewards', y_title='Proportion Place Fields', width=600)
fig.add_trace(go.Scattergl(x=rep_df['num_rewards'], y=rep_df['reward_zone'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color=ce_colors_dict[rep_df['group'].unique()[0]], name='Reward Area'))
fig.add_trace(go.Scattergl(x=rep_df['num_rewards'], y=rep_df['non_reward_zone'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color='darkgrey', name='Non-Reward Area'))
fig.add_trace(go.Scattergl(x=rep_df['num_rewards'], y=rep_df['other_port'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color='black', name=f'{port_type}'))
fig.update_yaxes(range=[-0.01, 0.12])
fig.show()

### Example mouse. This creates bins of size x number of rewards. This will allow combining across mice.

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'
port_type = 'Undershooting'
num_rewards_per_bin = 10
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)

In [ ]:
## Load and process data
output_dict = {'mouse': [], 'group': [], 'sex': [], 'bin': [], 'reward_zone': [], 'non_reward_zone': [], 'other_port': [], 'num_rewards': []}
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata.assign_coords(cumulative_rewards=('frame', np.cumsum(sdata['water'].values))) ## create cumulative rewards coordinate for later
cumulative_rewards = sdata['cumulative_rewards'].values
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Get positions of the under or over-shooting ports and use those as the zero location
front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
if port_type == 'Overshooting':
    pt1_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
    pt2_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
elif port_type == 'Undershooting':
    pt1_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
    pt2_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)

for rw_block in np.arange(0, math.ceil(np.sum(sdata['water'].values) / num_rewards_per_bin)):
    sub_bool = (cumulative_rewards >= rw_block * num_rewards_per_bin) & (cumulative_rewards < (rw_block + 1) * num_rewards_per_bin)
    sub_data = sdata[:, sub_bool]
    neural_data, position_data = ctn.subset_correct_dir_and_running(sub_data, correct_dir=correct_dir, only_running=only_running, 
                                                                    velocity_thresh=velocity_thresh)
    ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
    population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
    active_cells = np.sum(population_activity, axis=0) != 0
    population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
    tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
    tuning_curves = tuning_curves.T ## cells x spatial bin

    ## Find the peak of each place field
    pf_peaks = np.max(tuning_curves, axis=1)
    ## Find the spatial bins where each peak occurred
    field_dist = np.zeros(pf_peaks.shape[0])
    for idx, peak in enumerate(pf_peaks):
        field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]


    ## Make Reward 1 the first rewarding port the mouse got water from
    first_rew = sdata['lick_port'][sdata['water']].values[0]
    if first_rew == sdata.attrs['reward_one']:
        first_rw_pos = reward_one_pos 
        second_rw_pos = reward_two_pos
    else:
        first_rw_pos = reward_two_pos 
        second_rw_pos = reward_one_pos

    ## Find distance from reward locations
    rw_one_dist = field_dist - first_rw_pos
    rw_two_dist = field_dist - second_rw_pos
    bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
    sub_uids = neural_data['unit_id'][active_cells]
    rw1_uids = sub_uids['unit_id'][rw1_bool]
    rw2_uids = sub_uids['unit_id'][rw2_bool]

    ## Find distance from a an area close by
    ## Could play around by changing the index in all_midpoints, I chose 3 bins away because
    ## the build up of place fields can be wide
    non_rw_bin_start, non_rw_bin_end = all_midpoints[-1] - (reward_bin_size / 2), all_midpoints[-1] + (reward_bin_size / 2)
    non_rw1_bool = (rw_one_dist >= non_rw_bin_start) & (rw_one_dist < non_rw_bin_end)
    non_rw2_bool = (rw_two_dist >= non_rw_bin_start) & (rw_two_dist < non_rw_bin_end)
    non_rw1_uids = sub_uids['unit_id'][non_rw1_bool]
    non_rw2_uids = sub_uids['unit_id'][non_rw2_bool]

    ## Find distance from the undershooting or overshooting port
    pt_one_dist = field_dist - pt1_rw_pos
    pt_two_dist = field_dist - pt2_rw_pos
    bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
    pt1_bool = (pt_one_dist >= bin_start) & (pt_one_dist < bin_end)
    pt2_bool = (pt_two_dist >= bin_start) & (pt_two_dist < bin_end)
    pt1_uids = sub_uids['unit_id'][pt1_bool]
    pt2_uids = sub_uids['unit_id'][pt2_bool]

    output_dict['mouse'].append(mouse)
    output_dict['group'].append(sdata.attrs['group'])
    output_dict['sex'].append(sdata.attrs['sex'])
    output_dict['bin'].append((rw_block + 1) * num_rewards_per_bin) ## to shift the bin from zero since the mouse gets 9 rewards in the zero bin
    output_dict['reward_zone'].append((rw1_uids.shape[0] + rw2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['non_reward_zone'].append((non_rw1_uids.shape[0] + non_rw2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['other_port'].append((pt1_uids.shape[0] + pt2_uids.shape[0]) / sub_uids.shape[0])
    output_dict['num_rewards'].append(np.max(sub_data['cumulative_rewards'].values))

rep_df = pd.DataFrame(output_dict)

In [ ]:
## Plot proportion of place fields across rewards
fig = pf.custom_graph_template(x_title='Number of Rewards', y_title='Proportion Place Fields', width=600)
fig.add_trace(go.Scattergl(x=rep_df['bin'], y=rep_df['reward_zone'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color=ce_colors_dict[rep_df['group'].unique()[0]], name='Reward Area'))
fig.add_trace(go.Scattergl(x=rep_df['bin'], y=rep_df['non_reward_zone'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color='darkgrey', name='Non-Reward Area'))
fig.add_trace(go.Scattergl(x=rep_df['bin'], y=rep_df['other_port'], mode='lines+markers',
                           marker_size=9, marker=dict(line=dict(width=1.5, color='black')),
                           line_color='black', name=f'{port_type}'))
fig.update_yaxes(range=[-0.01, 0.12])
fig.show()

### Combine across mice.